# Building a house (Harvard Business Review, 1963)

Several tasks must be completed to build a house. Each task has a duration, and some tasks can start only after their predecessors have finished. Tasks may be worked on simultaneously; this model assumes sufficient workers and equipment for all tasks that are ready. How fast can the house be built?

Open the course repository root in VS Code, select the Julia 1.12 kernel with the course environment, and choose **Run All**. JuMP, HiGHS, and Plots are already included. All task data are included below; no external files are needed.


## Problem data

Durations are in days. Tasks `a` and `x` are zero-duration start and finish milestones.

| Task | Duration (days) | Immediate predecessors |
| --- | --- | --- |
| a | 0 | — |
| b | 4 | a |
| c | 2 | b |
| d | 4 | c |
| e | 6 | d |
| f | 1 | c |
| g | 2 | f |
| h | 3 | f |
| i | 2 | d |
| j | 4 | d, g |
| k | 10 | i, j, h |
| l | 3 | k |
| m | 1 | l |
| n | 2 | l |
| o | 3 | l |
| p | 2 | e |
| q | 1 | p |
| r | 1 | c |
| s | 2 | o, t |
| t | 3 | m, n |
| u | 1 | t |
| v | 2 | q, r |
| w | 5 | v |
| x | 0 | s, u, w |


In [ ]:
tasks = Symbol.('a':'x')

# Task durations in days.
dur = [0, 4, 2, 4, 6, 1, 2, 3, 2, 4, 10, 3, 1, 2, 3, 2, 1, 1, 2, 3, 1, 2, 5, 0]
duration = Dict(zip(tasks,dur))

# Immediate predecessors of each task.
pre = ( Symbol[], [:a], [:b], [:c], [:d], [:c], [:f], [:f], [:d], [:d,:g], [:i,:j,:h], [:k],
    [:l], [:l], [:l], [:e], [:p], [:c], [:o,:t], [:m,:n], [:t], [:q,:r], [:v], [:s,:u,:w])
pred = Dict(zip(tasks,pre));


In [ ]:
pred

## Mathematical Model

Decision variables:  
Start time $t_i$ of task $i$ for all tasks $i \in \mathcal{T}$, where $\mathcal{T}$ is the set of all tasks.
Note that we define a finish task, which defines the last "task" and has duration 0.

Constraints:  
Each task can only start when the previous task is done,  
$$t_i \geq t_j + d_j\quad \forall j \in  \mathcal{P}_i \quad \forall i \in\mathcal{T}$$
where $\mathcal{P}_i$ is the set of all tasks that must be finished before task $i$ can start (i.e., predecessor tasks) and $d_j$ is the duration of task $j$.

The first task will start at time 0, i.e.,
$$t_a = 0$$

Objective:  
$$\min_{t} t_x + d_x$$
where $t_x$ and $d_x$ are the start time and duration of the finish task, respectively. 

Every task lies on a path from `a` to `x`, so the precedence constraints and $t_a=0$ imply nonnegative start times and ensure every task finishes by $t_x$. The finish milestone depends on all terminal activities.


## JuMP Implementation

In [ ]:
using JuMP, HiGHS
import MathOptInterface as MOI
m = Model(HiGHS.Optimizer)
set_silent(m)  # Remove this line to see the solver log.

@variable(m, tstart[tasks])

# one-line implementation of the constraints:
# @constraint(m, link[i in tasks, j in pred[i]], tstart[i] >= tstart[j] + duration[j])

for i in tasks
    for j in pred[i]
        @constraint(m, tstart[i] >= tstart[j] + duration[j])
    end
end
@constraint(m, tstart[:a] == 0)
@objective(m, Min, tstart[:x] + duration[:x])  # Completion time of the finish milestone.

optimize!(m)
if termination_status(m) != MOI.OPTIMAL || primal_status(m) != MOI.FEASIBLE_POINT
    error("No optimal primal solution: $(termination_status(m)), $(primal_status(m))")
end
println(value.(tstart))
println("minimum duration: ", objective_value(m))


In [ ]:
println("Precedence-constraint slack (not task float):")
for i in tasks
    for j in pred[i]
       slack = value(tstart[i]) - value(tstart[j]) - duration[j];
        if (slack <= .00001)
            println("task ", i, " predecessor ",j, "  slack is : ", slack," ***ACTIVE")
        else
             println("task ", i, " predecessor ",j, "  slack is : ", slack)
        end
    end
end

## Critical tasks and total float

A tight precedence constraint is not enough to identify a critical task. Compute earliest starts by a forward pass and latest starts by a backward pass at the optimal completion time. Total float is latest minus earliest start; zero-float activities are critical. The dependency order below is computed from the graph, since alphabetical order is not a valid dependency order.


In [ ]:
# Forward pass: earliest starts and a dependency order.
earliest = Dict{Symbol,Float64}()
order = Symbol[]
while length(order) < length(tasks)
    ready = [i for i in tasks if !haskey(earliest, i) &&
             all(haskey(earliest, j) for j in pred[i])]
    isempty(ready) && error("The precedence graph contains a cycle")
    for i in ready
        earliest[i] = isempty(pred[i]) ? 0.0 :
            maximum(earliest[j] + duration[j] for j in pred[i])
        push!(order, i)
    end
end

# Backward pass: latest starts consistent with the minimum makespan.
latest = Dict{Symbol,Float64}()
for i in reverse(order)
    successors = [j for j in tasks if i in pred[j]]
    latest[i] = isempty(successors) ? objective_value(m) - duration[i] :
        minimum(latest[j] - duration[i] for j in successors)
end

total_float = Dict(i => latest[i] - earliest[i] for i in tasks)
critical = [i for i in order if abs(total_float[i]) <= 1e-7]
println("Critical activities (including dummy start/finish): ", critical)
println("Task m total float: ", total_float[:m], " day")


## Gantt chart: earliest-start schedule

Each solid bar runs from a task's earliest start to its earliest finish. Critical tasks are red, other tasks are blue, and the zero-duration start/finish milestones are diamonds. Rows are ordered by earliest start time.

The pale extensions show **total float**: the difference between earliest and latest finish at the optimal project completion time. They are not extra work. Using a task's float may require shifting its successors; the delays cannot generally be chosen independently. This chart uses the earliest-start schedule computed above, since the LP may return a different optimal schedule for noncritical tasks.


In [ ]:
using Plots
gr()

plot_tasks = sort(tasks; by=i -> (earliest[i], string(i)))
makespan = objective_value(m)
row_labels = [i == :a ? "a (start)" : i == :x ? "x (finish)" :
              string(i) for i in plot_tasks]
gantt_plot = plot(
    xlabel="Time (days)", ylabel="Task",
    yticks=(1:length(plot_tasks), row_labels), yflip=true,
    ylims=(0.4, length(plot_tasks) + 0.6),
    xlims=(-0.5, makespan + 1), xticks=0:2:ceil(Int, makespan),
    legend=:outerright, size=(1050, 800), left_margin=5 * Plots.mm,
    title="House construction: earliest-start schedule",
    grid=:x)

# Rectangles use actual start and finish times as their horizontal limits.
bar_rectangle(start, finish, row) = Shape(
    [start, finish, finish, start],
    [row - 0.3, row - 0.3, row + 0.3, row + 0.3])
shown_labels = Set{String}()
for (row, i) in enumerate(plot_tasks)
    start = earliest[i]
    finish = start + duration[i]
    if duration[i] == 0
        label = "Milestone" in shown_labels ? "" : "Milestone"
        scatter!(gantt_plot, [start], [row];
            marker=:diamond, color=:black, markersize=5, label=label)
        push!(shown_labels, "Milestone")
    else
        category = i in critical ? "Critical task" : "Noncritical task"
        color = i in critical ? :firebrick : :steelblue
        label = category in shown_labels ? "" : category
        plot!(gantt_plot, bar_rectangle(start, finish, row);
            fillcolor=color, linecolor=color, label=label)
        push!(shown_labels, category)
        if total_float[i] > 1e-7
            label = "Total float" in shown_labels ? "" : "Total float"
            plot!(gantt_plot,
                bar_rectangle(finish, latest[i] + duration[i], row);
                fillcolor=:lightblue, linecolor=:steelblue,
                linestyle=:dash, label=label)
            push!(shown_labels, "Total float")
        end
    end
end
vline!(gantt_plot, [makespan]; color=:gray, linestyle=:dash,
    label="Completion: $(round(makespan; digits=1)) days")
gantt_plot
